# Робота з моделями родини Meta

## Вступ

Цей урок охоплює:

- Ознайомлення з двома основними моделями родини Meta — Llama 3.1 та Llama 3.2, а також з наразі доступними альтернативами, які використовуються у прикладах коду (Llama 3.3 та Llama 4)
- Розуміння сценаріїв та випадків використання кожної моделі
- Приклад коду, що демонструє унікальні особливості кожної моделі


## Родина моделей Meta

У цьому уроці ми розглянемо 2 моделі з родини Meta або "Рій Llama" — Llama 3.1 та Llama 3.2. Запускаємі приклади коду використовують Llama 3.3 та Llama 4, які наразі є доступними альтернативами в каталозі Microsoft Foundry Models.

Ці моделі мають різні варіанти і доступні в [каталозі Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Примітка:** GitHub Models припиняє роботу наприкінці липня 2026 року. Детальніше про використання [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) для прототипування моделей ШІ.

> **Примітка:** Конкретні версії моделей, на яких спочатку базувався цей урок (Llama 3.1 70B/405B Instruct, Llama 3.2 11B/90B Vision Instruct), були зняті з каталогу Microsoft Foundry Models. Приклади коду нижче використовують `Llama-3.3-70B-Instruct` та `Llama-4-Scout-17B-16E-Instruct` як наразі доступні альтернативи в каталозі Microsoft Foundry Models, тож вони досі працюють.

Варіанти моделей (як описано у початковому уроці):
- Llama 3.1 - 70B Instruct (знято з використання)
- Llama 3.1 - 405B Instruct (знято з використання)
- Llama 3.2 - 11B Vision Instruct (знято з використання)
- Llama 3.2 - 90B Vision Instruct (знято з використання)

Поточні доступні альтернативи, які використовуються у прикладах коду:
- Llama 3.3 - 70B Instruct (`Llama-3.3-70B-Instruct`)
- Llama 4 Scout - 17B 16E Instruct (`Llama-4-Scout-17B-16E-Instruct`)

*Примітка: Варіанти Llama 3, описані вище, збережені для історичного контексту і більше не доступні у Microsoft Foundry Models.*



## Llama 3.1 

З 405 млрд параметрів, Llama 3.1 належить до категорії відкритих LLM. 

Модель є оновленням попереднього релізу Llama 3, пропонуючи: 

- Більше контекстне вікно - 128 тис. токенів проти 8 тис. токенів 
- Більша максимальна кількість вихідних токенів - 4096 проти 2048 
- Краща багатомовна підтримка - завдяки збільшенню кількості токенів для навчання 

Це дає змогу Llama 3.1 обробляти складніші випадки використання при створенні GenAI-додатків, включаючи: 
- Рідний виклик функцій - можливість викликати зовнішні інструменти та функції поза робочим процесом LLM
- Кращу продуктивність RAG - завдяки більшому контекстному вікну 
- Генерацію синтетичних даних - можливість створювати ефективні дані для таких завдань, як тонке налаштування 



### Виклик нативних функцій 

Llama 3.1 була донавчена для більш ефективного виклику функцій або інструментів. Вона також має два вбудовані інструменти, які модель може визначити як необхідні для використання на основі запиту користувача. Ці інструменти: 

- **Brave Search** - Можна використовувати для отримання актуальної інформації, наприклад, погоди, за допомогою веб-пошуку 
- **Wolfram Alpha** - Можна використовувати для більш складних математичних обчислень, тож написання власних функцій не потрібне. 

Ви також можете створювати власні інструменти, які LLM зможе викликати. 

У наведеному нижче прикладі коду: 

- Ми визначаємо доступні інструменти (brave_search, wolfram_alpha) у системному запиті. 
- Відправляємо користувацький запит, який запитує про погоду у певному місті. 
- LLM відповість викликом інструменту Brave Search, який виглядатиме так `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Примітка: цей приклад лише здійснює виклик інструменту, якщо ви хочете отримати результати, вам потрібно створити безкоштовний акаунт на сторінці Brave API та визначити саму функцію` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Meta-Llama-3.1-405B-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-3.3-70B-Instruct, a currently available alternative, instead
model_name = "Llama-3.3-70B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Незважаючи на те, що Llama 3.1 є великою мовною моделлю (LLM), одним з її обмежень була мультимодальність. Тобто, можливість використовувати різні типи вхідних даних, такі як зображення в якості підказок та надавати відповіді. Ця можливість є однією з основних особливостей Llama 3.2. Ці особливості також включають: 

- Мультимодальність - здатність оцінювати як текстові, так і графічні підказки 
- Варіанти малого та середнього розміру (11B та 90B) - це забезпечує гнучкі варіанти розгортання, 
- Тільки текстові варіанти (1B та 3B) - це дозволяє розгортати модель на edge / мобільних пристроях і забезпечує низьку затримку 

Підтримка мультимодальності є великим кроком у світі моделей з відкритим кодом. Наведений нижче приклад коду приймає як зображення, так і текстову підказку для отримання аналізу зображення. Оскільки Llama 3.2 90B Vision Instruct було знято з експлуатації, у прикладі використовується `Llama-4-Scout-17B-16E-Instruct`, наразі доступна нативна мультимодальна альтернатива. 

### Підтримка мультимодальності з Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Llama-3.2-90B-Vision-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-4-Scout-17B-16E-Instruct, a currently available natively multimodal alternative, instead
model_name = "Llama-4-Scout-17B-16E-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Навчання на цьому не закінчується, продовжуйте подорож

Після завершення цього уроку ознайомтеся з нашою [колекцією навчання з генеративного ШІ](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst), щоб продовжувати поглиблювати свої знання про генеративний ШІ!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
